<a href="https://colab.research.google.com/github/gustavopatinot/SECOP_Vec_Local/blob/main/notebooks_colab/generador_vectores_y_verdad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Librerías necesarias para utilizar en GPU

In [ ]:
!pip install -q sentence-transformers pyarrow numpy torch

# Módulo para ejecutar en GPU

In [ ]:
# -*- coding: utf-8 -*-
"""
MÓDULO 02 (AUTÓNOMO): GENERADOR DE EMBEDDINGS Y VERDAD DE REFERENCIA EN GPU
================================================================================
Proyecto de Tesis de Maestría:
    Búsqueda semántica a gran escala con memoria restringida:
    HNSW cuantizado aplicado a la contratación pública colombiana (SECOP II)
    en un computador de 8 GB de RAM.

Objetivo Específico 1 y Línea Base de Hipótesis H3:
    1. Montar Google Drive de forma autónoma y verificar la integridad de los
       archivos generados en la Etapa 1 (N = 1.990.000 docs y |Q| = 10.000 consultas).
    2. Generar los embeddings con multilingual-e5-base (768 dimensiones) usando
       checkpoints en disco (memmap float16) resistentes a desconexiones.
    3. Calcular por fuerza bruta en GPU los top-100 vecinos exactos para las
       10.000 consultas y medir el techo exacto de precisión@10 y nDCG@10 por
       jerarquía UNSPSC (segmento, familia y clase).

Entorno de Ejecución:
    Instancia independiente de Google Colab Pro con GPU + Google Drive.
================================================================================
"""

import os
import json
import time
import logging
from pathlib import Path
from typing import Dict, List

import numpy as np
import pyarrow.parquet as pq
import torch
from sentence_transformers import SentenceTransformer

# Configuración del registro de auditoría científica en español
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | [%(levelname)s] | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)
registro_auditoria = logging.getLogger("Modulo02AutonomoGPU")


class ConfiguracionEtapa2GPU:
    """Parámetros técnicos autónomos para la Etapa 2 en Google Colab Pro."""

    # 1. Modelo multilingüe local definido en la metodología de la tesis
    NOMBRE_MODELO_LOCAL: str = "intfloat/multilingual-e5-base"
    DIMENSION_BASE: int = 768
    LONGITUD_MAXIMA_TOKENS: int = 160  # Cubre media=251.51 y mediana=234.0 caracteres

    # 2. Parámetros de procesamiento por lotes en GPU
    LOTE_INFERENCIA_GPU: int = 1024
    LOTE_CHECKPOINT_DRIVE: int = 50_000
    BLOQUE_FUERZA_BRUTA_GPU: int = 100_000
    TOP_K_VERDAD_EXACTA: int = 100

    # 3. Estructura de carpetas persistentes en Google Drive
    RUTA_RAIZ_DRIVE: Path = Path("/content/drive/MyDrive/SECOP_Vec_Tesis")
    RUTA_ORO: Path = RUTA_RAIZ_DRIVE / "03_oro_benchmark_secop_vec"
    RUTA_REPORTES_TESIS: Path = RUTA_RAIZ_DRIVE / "04_tablas_y_metricas_tesis"

    # 4. Archivos de entrada creados en la Etapa 1
    ARCHIVO_CORPUS_PARQUET: Path = RUTA_ORO / "secop_vec_corpus_base.parquet"
    ARCHIVO_CONSULTAS_PARQUET: Path = RUTA_ORO / "secop_vec_consultas_10k.parquet"
    ARCHIVO_ESTADISTICAS_ETAPA1: Path = RUTA_REPORTES_TESIS / "tabla_3_1_estadisticas_corpus.json"

    # 5. Archivos binarios de salida (Embeddings y Checkpoints)
    ARCHIVO_VECTORES_CORPUS_F16: Path = RUTA_ORO / "vectores_corpus_e5_768d_f16.mmap"
    ARCHIVO_VECTORES_CONSULTAS_F32: Path = RUTA_ORO / "vectores_consultas_e5_768d_f32.npy"
    ARCHIVO_CHECKPOINT_VECTORES: Path = RUTA_ORO / "estado_vectorizacion_e5.json"

    # 6. Archivos de salida de la Verdad de Referencia Exacta (Ground Truth)
    ARCHIVO_VERDAD_IDS_TOP100: Path = RUTA_ORO / "verdad_exacta_top100_indices.npy"
    ARCHIVO_VERDAD_SCORES_TOP100: Path = RUTA_ORO / "verdad_exacta_top100_similitud.npy"
    ARCHIVO_REPORTE_TABLA_3_2: Path = RUTA_REPORTES_TESIS / "tabla_3_2_verdad_referencia_gpu.json"


def inicializar_entorno_gpu_y_drive(config: ConfiguracionEtapa2GPU) -> Dict:
    """
    Monta Google Drive en la nueva sesión de Colab, verifica que la GPU esté
    activa y valida que los archivos Parquet de la Etapa 1 existan.
    """
    # 1. Montaje autónomo de Google Drive
    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            registro_auditoria.info("Montando Google Drive en el nuevo entorno con GPU...")
            drive.mount("/content/drive")
        else:
            registro_auditoria.info("Google Drive ya se encuentra montado.")
    except ImportError:
        registro_auditoria.warning("Entorno local detectado (fuera de Google Colab).")

    # 2. Verificación estricta de artefactos de la Etapa 1
    for archivo_requerido in [config.ARCHIVO_CORPUS_PARQUET, config.ARCHIVO_CONSULTAS_PARQUET]:
        if not archivo_requerido.exists():
            raise FileNotFoundError(
                f"No se encontró el archivo de la Etapa 1 en Drive: {archivo_requerido}. "
                f"Verifica que hayas montado la misma cuenta de Google Drive."
            )

    config.RUTA_REPORTES_TESIS.mkdir(parents=True, exist_ok=True)

    # 3. Verificación de hardware GPU
    if not torch.cuda.is_available():
        raise RuntimeError(
            "ERROR: No hay GPU activa en este entorno. Ve a 'Entorno de ejecución' -> "
            "'Cambiar tipo de entorno de ejecución' y selecciona una GPU (T4, L4 o A100)."
        )

    nombre_gpu = torch.cuda.get_device_name(0)
    memoria_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    num_docs = pq.read_metadata(config.ARCHIVO_CORPUS_PARQUET).num_rows
    num_consultas = pq.read_metadata(config.ARCHIVO_CONSULTAS_PARQUET).num_rows

    info_entorno = {
        "gpu_activa": nombre_gpu,
        "vram_total_gb": round(memoria_vram_gb, 2),
        "documentos_corpus_detectados": int(num_docs),
        "consultas_detectadas": int(num_consultas),
    }
    registro_auditoria.info(
        f"Entorno validado | GPU: {nombre_gpu} ({memoria_vram_gb:.1f} GB VRAM) | "
        f"Corpus: {num_docs:,} docs | Consultas: {num_consultas:,}"
    )
    return info_entorno


class MotorVectorizacionSECOPVec:
    """
    Genera los embeddings asimétricos ('query: ' y 'passage: ') con multilingual-e5-base
    en precisión FP16 y los persiste en Google Drive mediante numpy.memmap con checkpoints.
    """

    def __init__(self, config: ConfiguracionEtapa2GPU):
        self.config = config
        self.dispositivo = "cuda"
        registro_auditoria.info(f"Cargando modelo '{self.config.NOMBRE_MODELO_LOCAL}' en GPU...")
        self.modelo = SentenceTransformer(
            self.config.NOMBRE_MODELO_LOCAL,
            device=self.dispositivo,
        )
        self.modelo.max_seq_length = self.config.LONGITUD_MAXIMA_TOKENS
        self.modelo.half()  # Aceleración en Tensor Cores FP16

    def _leer_progreso(self) -> int:
        if self.config.ARCHIVO_CHECKPOINT_VECTORES.exists():
            with open(self.config.ARCHIVO_CHECKPOINT_VECTORES, "r", encoding="utf-8") as f:
                return int(json.load(f).get("documentos_vectorizados", 0))
        return 0

    def _guardar_progreso(self, cantidad: int, total: int, segundos_acumulados: float) -> None:
        estado = {
            "documentos_vectorizados": cantidad,
            "total_documentos": total,
            "porcentaje_completado": round(100.0 * cantidad / total, 2),
            "segundos_acumulados": round(segundos_acumulados, 2),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        archivo_tmp = self.config.ARCHIVO_CHECKPOINT_VECTORES.with_suffix(".tmp")
        with open(archivo_tmp, "w", encoding="utf-8") as f:
            json.dump(estado, f, indent=2, ensure_ascii=False)
        archivo_tmp.replace(self.config.ARCHIVO_CHECKPOINT_VECTORES)

    def ejecutar_vectorizacion(self) -> Dict[str, float]:
        t_inicio_global = time.perf_counter()

        # 1. Vectorización de las 10.000 consultas
        if not self.config.ARCHIVO_VECTORES_CONSULTAS_F32.exists():
            registro_auditoria.info("Codificando las 10.000 consultas con prefijo 'query: '...")
            tabla_q = pq.read_table(
                self.config.ARCHIVO_CONSULTAS_PARQUET, columns=["descripcion_objeto"]
            )
            textos_q = [
                f"query: {texto}"
                for texto in tabla_q.column("descripcion_objeto").to_pylist()
            ]
            vectores_q = self.modelo.encode(
                textos_q,
                batch_size=self.config.LOTE_INFERENCIA_GPU,
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=True,
            ).astype(np.float32)
            np.save(self.config.ARCHIVO_VECTORES_CONSULTAS_F32, vectores_q)
            registro_auditoria.info(f"Vectores de consultas guardados: {vectores_q.shape}")
        else:
            registro_auditoria.info("Los vectores de las 10.000 consultas ya existen en Drive.")

        # 2. Vectorización incremental de los 1.990.000 documentos del corpus
        tabla_c = pq.read_table(
            self.config.ARCHIVO_CORPUS_PARQUET, columns=["descripcion_objeto"]
        )
        total_docs = tabla_c.num_rows
        inicio_offset = self._leer_progreso()

        if inicio_offset >= total_docs and self.config.ARCHIVO_VECTORES_CORPUS_F16.exists():
            registro_auditoria.info(
                f"El corpus completo ({total_docs:,} documentos) ya fue vectorizado previamente."
            )
        else:
            modo_apertura = (
                "r+"
                if (inicio_offset > 0 and self.config.ARCHIVO_VECTORES_CORPUS_F16.exists())
                else "w+"
            )
            matriz_corpus_mmap = np.memmap(
                self.config.ARCHIVO_VECTORES_CORPUS_F16,
                dtype=np.float16,
                mode=modo_apertura,
                shape=(total_docs, self.config.DIMENSION_BASE),
            )

            registro_auditoria.info(
                f"Vectorizando corpus desde el documento {inicio_offset:,} hasta {total_docs:,}..."
            )
            columna_textos = tabla_c.column("descripcion_objeto")
            paso = self.config.LOTE_CHECKPOINT_DRIVE

            for inicio in range(inicio_offset, total_docs, paso):
                fin = min(inicio + paso, total_docs)
                t_lote = time.perf_counter()

                textos_lote = [
                    f"passage: {t}"
                    for t in columna_textos.slice(inicio, fin - inicio).to_pylist()
                ]
                emb_lote = self.modelo.encode(
                    textos_lote,
                    batch_size=self.config.LOTE_INFERENCIA_GPU,
                    normalize_embeddings=True,
                    convert_to_numpy=True,
                    show_progress_bar=False,
                ).astype(np.float16)

                matriz_corpus_mmap[inicio:fin, :] = emb_lote
                matriz_corpus_mmap.flush()

                segundos_lote = time.perf_counter() - t_lote
                velocidad = (fin - inicio) / max(segundos_lote, 1e-6)
                self._guardar_progreso(fin, total_docs, time.perf_counter() - t_inicio_global)

                registro_auditoria.info(
                    f"Checkpoint Drive: {fin:,}/{total_docs:,} ({100.0 * fin / total_docs:.1f}%) | "
                    f"Velocidad GPU: {velocidad:,.0f} docs/s"
                )

        tamano_gb = self.config.ARCHIVO_VECTORES_CORPUS_F16.stat().st_size / (1024 ** 3)
        return {
            "total_documentos_vectorizados": int(total_docs),
            "dimension_embeddings": self.config.DIMENSION_BASE,
            "tamano_archivo_mmap_float16_gb": round(tamano_gb, 3),
            "tiempo_sesion_minutos": round((time.perf_counter() - t_inicio_global) / 60.0, 2),
        }


class CalculadorVerdadReferenciaExactaGPU:
    """
    Calcula por fuerza bruta en GPU los top-100 vecinos exactos para las 10.000 consultas
    y evalúa las métricas exactas de relevancia por código UNSPSC (precisión@10 y nDCG@10).
    """

    def __init__(self, config: ConfiguracionEtapa2GPU):
        self.config = config
        self.dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    @torch.no_grad()
    def ejecutar_fuerza_bruta_y_metricas_base(
        self, dimension_evaluacion: int = 768
    ) -> Dict[str, float]:
        t0 = time.perf_counter()

        vectores_q_np = np.load(self.config.ARCHIVO_VECTORES_CONSULTAS_F32)[:, :dimension_evaluacion]
        vectores_q_np /= np.linalg.norm(vectores_q_np, axis=1, keepdims=True) + 1e-12

        num_consultas = int(vectores_q_np.shape[0])
        k = self.config.TOP_K_VERDAD_EXACTA

        bytes_totales = self.config.ARCHIVO_VECTORES_CORPUS_F16.stat().st_size
        total_docs = int(bytes_totales // (self.config.DIMENSION_BASE * 2))

        matriz_corpus_mmap = np.memmap(
            self.config.ARCHIVO_VECTORES_CORPUS_F16,
            dtype=np.float16,
            mode="r",
            shape=(total_docs, self.config.DIMENSION_BASE),
        )

        tensor_q = torch.from_numpy(vectores_q_np).to(self.dispositivo, dtype=torch.float16)

        mejores_scores = torch.full(
            (num_consultas, k), -1e4, device=self.dispositivo, dtype=torch.float16
        )
        mejores_indices = torch.full(
            (num_consultas, k), -1, device=self.dispositivo, dtype=torch.int64
        )

        tamano_bloque = self.config.BLOQUE_FUERZA_BRUTA_GPU
        registro_auditoria.info(
            f"Iniciando fuerza bruta exacta en GPU ({num_consultas:,} consultas x {total_docs:,} docs)..."
        )

        for inicio in range(0, total_docs, tamano_bloque):
            fin = min(inicio + tamano_bloque, total_docs)
            bloque_np = np.array(
                matriz_corpus_mmap[inicio:fin, :dimension_evaluacion], copy=True
            )
            tensor_bloque = torch.from_numpy(bloque_np).to(
                self.dispositivo, dtype=torch.float16
            )
            if dimension_evaluacion < self.config.DIMENSION_BASE:
                tensor_bloque = torch.nn.functional.normalize(tensor_bloque, p=2, dim=1)

            similitudes_bloque = torch.matmul(tensor_q, tensor_bloque.T)
            scores_bloque, idx_locales = torch.topk(
                similitudes_bloque, k=k, dim=1, largest=True, sorted=True
            )
            idx_globales = idx_locales + inicio

            scores_combinados = torch.cat([mejores_scores, scores_bloque], dim=1)
            indices_combinados = torch.cat([mejores_indices, idx_globales], dim=1)

            mejores_scores, pos_ganadoras = torch.topk(
                scores_combinados, k=k, dim=1, largest=True, sorted=True
            )
            mejores_indices = torch.gather(indices_combinados, 1, pos_ganadoras)

            del tensor_bloque, similitudes_bloque
            registro_auditoria.info(
                f"Fuerza bruta GPU: {fin:,}/{total_docs:,} documentos procesados."
            )

        indices_top100 = mejores_indices.cpu().numpy().astype(np.int32)
        scores_top100 = mejores_scores.cpu().numpy().astype(np.float32)

        np.save(self.config.ARCHIVO_VERDAD_IDS_TOP100, indices_top100)
        np.save(self.config.ARCHIVO_VERDAD_SCORES_TOP100, scores_top100)
        tiempo_fuerza_bruta = time.perf_counter() - t0

        # Cálculo de la relevancia exacta de referencia por código UNSPSC (Techo para H3)
        metricas_unspsc = self._evaluar_relevancia_unspsc_top10(indices_top100[:, :10])

        resumen_tabla_3_2 = {
            "consultas_evaluadas": num_consultas,
            "documentos_corpus": total_docs,
            "top_k_exacto": int(k),
            "similitud_coseno_media_top1": round(float(np.mean(scores_top100[:, 0])), 4),
            "similitud_coseno_media_top10": round(float(np.mean(scores_top100[:, :10])), 4),
            "similitud_coseno_media_top100": round(float(np.mean(scores_top100)), 4),
            "precision_at_10_exacta_segmento": metricas_unspsc["precision_at_10_segmento"],
            "precision_at_10_exacta_familia": metricas_unspsc["precision_at_10_familia"],
            "precision_at_10_exacta_clase": metricas_unspsc["precision_at_10_clase"],
            "ndcg_at_10_exacto_jerarquico": metricas_unspsc["ndcg_at_10_jerarquico"],
            "tiempo_calculo_fuerza_bruta_segundos": round(tiempo_fuerza_bruta, 2),
        }

        with open(self.config.ARCHIVO_REPORTE_TABLA_3_2, "w", encoding="utf-8") as f:
            json.dump(resumen_tabla_3_2, f, indent=2, ensure_ascii=False)

        registro_auditoria.info(
            f"Tabla 3.2 exportada exitosamente en: {self.config.ARCHIVO_REPORTE_TABLA_3_2}"
        )
        return resumen_tabla_3_2

    def _evaluar_relevancia_unspsc_top10(self, indices_top10: np.ndarray) -> Dict[str, float]:
        """
        Evalúa precisión@10 por nivel UNSPSC y nDCG@10 jerárquico (ganancia 0, 1, 2, 3)
        sobre los vecinos exactos recuperados para establecer la línea base de H3.
        """
        registro_auditoria.info("Calculando línea base de relevancia UNSPSC sobre búsqueda exacta...")
        columnas_unspsc = ["unspsc_segmento", "unspsc_familia", "unspsc_clase"]

        tabla_c = pq.read_table(self.config.ARCHIVO_CORPUS_PARQUET, columns=columnas_unspsc)
        tabla_q = pq.read_table(self.config.ARCHIVO_CONSULTAS_PARQUET, columns=columnas_unspsc)

        seg_c = np.array(tabla_c.column("unspsc_segmento").to_pylist())
        fam_c = np.array(tabla_c.column("unspsc_familia").to_pylist())
        cla_c = np.array(tabla_c.column("unspsc_clase").to_pylist())

        seg_q = np.array(tabla_q.column("unspsc_segmento").to_pylist())[:, None]
        fam_q = np.array(tabla_q.column("unspsc_familia").to_pylist())[:, None]
        cla_q = np.array(tabla_q.column("unspsc_clase").to_pylist())[:, None]

        aciertos_seg = (seg_c[indices_top10] == seg_q).astype(np.float32)
        aciertos_fam = (fam_c[indices_top10] == fam_q).astype(np.float32)
        aciertos_cla = (cla_c[indices_top10] == cla_q).astype(np.float32)

        # Ganancia gradual rel in {0, 1, 2, 3} y descuento logarítmico para nDCG@10
        ganancia_gradual = aciertos_seg + aciertos_fam + aciertos_cla
        descuentos = 1.0 / np.log2(np.arange(2, 12, dtype=np.float32))
        dcg = np.sum((np.power(2.0, ganancia_gradual) - 1.0) * descuentos, axis=1)

        # Como en la Etapa 1 garantizamos que cada clase de consulta tiene >= 10 contratos,
        # el IDCG@10 ideal corresponde a 10 documentos de la misma clase (rel = 3 -> 2^3 - 1 = 7)
        idcg_ideal = float(np.sum(7.0 * descuentos))
        ndcg_jerarquico = float(np.mean(dcg / idcg_ideal))

        return {
            "precision_at_10_segmento": round(float(np.mean(aciertos_seg)), 4),
            "precision_at_10_familia": round(float(np.mean(aciertos_fam)), 4),
            "precision_at_10_clase": round(float(np.mean(aciertos_cla)), 4),
            "ndcg_at_10_jerarquico": round(ndcg_jerarquico, 4),
        }

# Ejecución módulo 2

In [ ]:
config_etapa2 = ConfiguracionEtapa2GPU()

# 1. Montaje autónomo de Drive y verificación de artefactos de la Etapa 1
diagnostico = inicializar_entorno_gpu_y_drive(config_etapa2)
print("=== DIAGNÓSTICO DEL ENTORNO GPU Y DRIVE ===")
print(json.dumps(diagnostico, indent=2, ensure_ascii=False))

# 2. Generación de embeddings (con checkpoints cada 50.000 documentos en Drive)
motor_embeddings = MotorVectorizacionSECOPVec(config_etapa2)
resumen_embeddings = motor_embeddings.ejecutar_vectorizacion()
print("\n=== RESUMEN DE VECTORIZACIÓN (SECOP-VEC v1) ===")
print(json.dumps(resumen_embeddings, indent=2, ensure_ascii=False))

# 3. Fuerza bruta exacta en GPU y línea base de relevancia UNSPSC
calculador_verdad = CalculadorVerdadReferenciaExactaGPU(config_etapa2)
tabla_3_2 = calculador_verdad.ejecutar_fuerza_bruta_y_metricas_base(dimension_evaluacion=768)
print("\n=== TABLA 3.2: VERDAD DE REFERENCIA EXACTA Y LÍNEA BASE UNSPSC ===")
print(json.dumps(tabla_3_2, indent=2, ensure_ascii=False))

2026-11-15 16:56:27 | [INFO] | Montando Google Drive en el nuevo entorno con GPU...


Mounted at /content/drive


2026-11-15 16:56:54 | [INFO] | Entorno validado | GPU: NVIDIA L4 (22.0 GB VRAM) | Corpus: 1,990,000 docs | Consultas: 10,000
2026-11-15 16:56:54 | [INFO] | Cargando modelo 'intfloat/multilingual-e5-base' en GPU...


=== DIAGNÓSTICO DEL ENTORNO GPU Y DRIVE ===
{
  "gpu_activa": "NVIDIA L4",
  "vram_total_gb": 22.03,
  "documentos_corpus_detectados": 1990000,
  "consultas_detectadas": 10000
}


2026-11-15 16:56:55 | [INFO] | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-11-15 16:56:55 | [INFO] | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-11-15 16:56:55 | [WARNING] | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-11-15 16:56:55 | [INFO] | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/d128750597153bb5987e10b1c3493a34e5a4502a/modules.json?%2Fintfloat%2Fmultilingual-e5-base%2Fresolve%2Fmain%2Fmodules.json=&etag=%22ac2039abdf6ff023b27c919bb9675cfe378cb10f%22 "HTTP/1.1 200 OK"
2026-11-15 16:56:55 | [INFO] | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/d128750597153bb5987e10b1c3493a34e5a4502a/modules.json?%2Fintfloat%2Fmultilingual-e5-base%2Fresolve%2Fmain%2F

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

... (24 outputs left out)

2026-11-15 16:57:22 | [INFO] | Vectores de consultas guardados: (10000, 768)
2026-11-15 16:57:36 | [INFO] | Vectorizando corpus desde el documento 0 hasta 1,990,000...
2026-11-15 16:58:10 | [INFO] | Checkpoint Drive: 50,000/1,990,000 (2.5%) | Velocidad GPU: 1,459 docs/s
2026-11-15 16:58:45 | [INFO] | Checkpoint Drive: 100,000/1,990,000 (5.0%) | Velocidad GPU: 1,453 docs/s
2026-11-15 16:59:21 | [INFO] | Checkpoint Drive: 150,000/1,990,000 (7.5%) | Velocidad GPU: 1,385 docs/s
2026-11-15 16:59:57 | [INFO] | Checkpoint Drive: 200,000/1,990,000 (10.1%) | Velocidad GPU: 1,371 docs/s
2026-11-15 17:00:34 | [INFO] | Checkpoint Drive: 250,000/1,990,000 (12.6%) | Velocidad GPU: 1,364 docs/s
2026-11-15 17:01:10 | [INFO] | Checkpoint Drive: 300,000/1,990,000 (15.1%) | Velocidad GPU: 1,379 docs/s
2026-11-15 17:01:47 | [INFO] | Checkpoint Drive: 350,000/1,990,000 (17.6%) | Velocidad GPU: 1,380 docs/s
2026-11-15 17:02:24 | [INFO] | Checkpoint Drive: 400,000/1,990,000 (20.1%) | Velocidad GPU: 1,354 doc


=== RESUMEN DE VECTORIZACIÓN (SECOP-VEC v1) ===
{
  "total_documentos_vectorizados": 1990000,
  "dimension_embeddings": 768,
  "tamano_archivo_mmap_float16_gb": 2.847,
  "tiempo_sesion_minutos": 24.48
}


2026-11-15 17:21:42 | [INFO] | Fuerza bruta GPU: 100,000/1,990,000 documentos procesados.
2026-11-15 17:21:42 | [INFO] | Fuerza bruta GPU: 200,000/1,990,000 documentos procesados.
2026-11-15 17:21:42 | [INFO] | Fuerza bruta GPU: 300,000/1,990,000 documentos procesados.
2026-11-15 17:21:43 | [INFO] | Fuerza bruta GPU: 400,000/1,990,000 documentos procesados.
2026-11-15 17:21:43 | [INFO] | Fuerza bruta GPU: 500,000/1,990,000 documentos procesados.
2026-11-15 17:21:43 | [INFO] | Fuerza bruta GPU: 600,000/1,990,000 documentos procesados.
2026-11-15 17:21:43 | [INFO] | Fuerza bruta GPU: 700,000/1,990,000 documentos procesados.
2026-11-15 17:21:43 | [INFO] | Fuerza bruta GPU: 800,000/1,990,000 documentos procesados.
2026-11-15 17:21:44 | [INFO] | Fuerza bruta GPU: 900,000/1,990,000 documentos procesados.
2026-11-15 17:21:44 | [INFO] | Fuerza bruta GPU: 1,000,000/1,990,000 documentos procesados.
2026-11-15 17:21:44 | [INFO] | Fuerza bruta GPU: 1,100,000/1,990,000 documentos procesados.
2026-1


=== TABLA 3.2: VERDAD DE REFERENCIA EXACTA Y LÍNEA BASE UNSPSC ===
{
  "consultas_evaluadas": 10000,
  "documentos_corpus": 1990000,
  "top_k_exacto": 100,
  "similitud_coseno_media_top1": 0.9433,
  "similitud_coseno_media_top10": 0.927,
  "similitud_coseno_media_top100": 0.9081,
  "precision_at_10_exacta_segmento": 0.7879,
  "precision_at_10_exacta_familia": 0.7145,
  "precision_at_10_exacta_clase": 0.6597,
  "ndcg_at_10_exacto_jerarquico": 0.7112,
  "tiempo_calculo_fuerza_bruta_segundos": 3.66
}


# verificacion datos en Google Drive

In [ ]:
!ls -lh /content/drive/MyDrive/SECOP_Vec_Tesis/03_oro_benchmark_secop_vec/ /content/drive/MyDrive/SECOP_Vec_Tesis/04_tablas_y_metricas_tesis/

/content/drive/MyDrive/SECOP_Vec_Tesis/03_oro_benchmark_secop_vec/:
total 3.1G
-rw------- 1 root root  177 Nov 15 17:21 estado_vectorizacion_e5.json
-rw------- 1 root root 910K Nov 15 16:31 secop_vec_consultas_10k.parquet
-rw------- 1 root root 167M Nov 15 16:31 secop_vec_corpus_base.parquet
-rw------- 1 root root  30M Nov 15 16:57 vectores_consultas_e5_768d_f32.npy
-rw------- 1 root root 2.9G Nov 15 17:21 vectores_corpus_e5_768d_f16.mmap
-rw------- 1 root root 3.9M Nov 15 17:21 verdad_exacta_top100_indices.npy
-rw------- 1 root root 3.9M Nov 15 17:21 verdad_exacta_top100_similitud.npy

/content/drive/MyDrive/SECOP_Vec_Tesis/04_tablas_y_metricas_tesis/:
total 1.0K
-rw------- 1 root root 370 Nov 15 16:31 tabla_3_1_estadisticas_corpus.json
-rw------- 1 root root 435 Nov 15 17:22 tabla_3_2_verdad_referencia_gpu.json
